# N01 — Pencarian Heuristik dengan A*

**Pendamping S03 — Kecerdasan Buatan & ML Dasar**

Pola kerja: **DO → OBSERVE → INTERPRET → EXPLAIN**. Jalankan sel secara berurutan, catat evidence, lalu tuliskan interpretasi Anda.

## Tujuan
Mengimplementasikan A* pada graph kecil dan membandingkan beberapa heuristic.

In [1]:
import heapq

graph = {
    'A': {'B': 2, 'C': 5},
    'B': {'C': 1, 'D': 4},
    'C': {'D': 1, 'E': 7},
    'D': {'E': 2},
    'E': {}
}
heuristic = {'A': 5, 'B': 4, 'C': 3, 'D': 2, 'E': 0}

def astar(graph, start, goal, h):
    frontier = [(h[start], 0, start, [start])]
    best_g = {start: 0}
    expanded = []
    while frontier:
        f, g, node, path = heapq.heappop(frontier)
        if g != best_g.get(node):
            continue
        expanded.append(node)
        if node == goal:
            return path, g, expanded
        for nxt, cost in graph[node].items():
            new_g = g + cost
            if new_g < best_g.get(nxt, float('inf')):
                best_g[nxt] = new_g
                heapq.heappush(frontier, (new_g + h[nxt], new_g, nxt, path + [nxt]))
    return None, float('inf'), expanded

path, cost, expanded = astar(graph, 'A', 'E', heuristic)
path, cost, expanded

(['A', 'B', 'C', 'D', 'E'], 6, ['A', 'B', 'C', 'D', 'E'])

### Eksperimen 1
Catat path, total cost, dan urutan node yang dieksplor. Jelaskan peran **g(n)**, **h(n)**, dan **f(n)=g(n)+h(n)**.

In [2]:
h_zero = {k: 0 for k in heuristic}
for name, h in [('heuristic_awal', heuristic), ('h=0', h_zero)]:
    path, cost, expanded = astar(graph, 'A', 'E', h)
    print(name, '=>', path, 'cost=', cost, 'expanded=', expanded)

heuristic_awal => ['A', 'B', 'C', 'D', 'E'] cost= 6 expanded= ['A', 'B', 'C', 'D', 'E']
h=0 => ['A', 'B', 'C', 'D', 'E'] cost= 6 expanded= ['A', 'B', 'C', 'D', 'E']


**Catatan**

path = A => B => C => D => E
cost = 6 [2 + 1 + 1 + 2]
urutan node yang dieksplor = A , B , C , D , E


Peran g(n), h(n), dan f(n)=g(n)+h(n)
1. G(n) – Actual Cost (Biaya Sebenarnya)*
   G(n) adalah total biaya yang sudah dikeluarkan dari titik awal (start) sampai ke node yang sedang dikunjungi.

2. H(n) – Heuristic Cost (Perkiraan Biaya)*
   H(n) adalah perkiraan biaya dari node yang sedang dikunjungi menuju node tujuan (goal).

3. F(n) – Total Estimated Cost (Total Perkiraan Biaya)*
   F(n) adalah hasil penjumlahan antara G(n) dan H(n). Nilai ini digunakan oleh algoritma A* untuk menentukan node mana yang diprioritaskan untuk diperiksa.


### Eksperimen 2
Bandingkan heuristic awal dengan **h=0**. Apakah solusi berubah? Apakah jumlah/urutan node yang dieksplor berubah?

In [3]:
h_zero = {k: 0 for k in heuristic}

for name, h in [('heuristic_awal', heuristic), ('h=0', h_zero)]:
    path, cost, expanded = astar(graph, 'A', 'E', h)
    print(name, '=>', path, 'cost=', cost, 'expanded=', expanded)

heuristic_awal => ['A', 'B', 'C', 'D', 'E'] cost= 6 expanded= ['A', 'B', 'C', 'D', 'E']
h=0 => ['A', 'B', 'C', 'D', 'E'] cost= 6 expanded= ['A', 'B', 'C', 'D', 'E']


**Hasil Eksperimen 2**

- **Heuristic awal:** Jalur A → B → C → D → E, total biaya = 6, jumlah node yang dieksplorasi = 5.
- **Heuristic h = 0:** Jalur A → B → C → D → E, total biaya = 6, jumlah node yang dieksplorasi = 5.

**Kesimpulan:** Pada eksperimen ini, perubahan heuristic menjadi nol tidak mengubah jalur, total biaya, jumlah node yang dieksplorasi, maupun urutan eksplorasi.

### Eksperimen 3 — Heuristic alternatif (uji risiko heuristic terlalu besar)

**Tujuan:** melihat apakah heuristic yang terlalu besar dapat mengubah rute dan kualitas solusi. Kode `astar()` dan `graph` asli tidak diubah; hanya salinan nilai heuristic yang diubah.

In [4]:
h_alternatif = heuristic.copy()
h_alternatif['C'] = 10  # nilai C diubah dari 3 menjadi 10

path_alt, cost_alt, expanded_alt = astar(graph, 'A', 'E', h_alternatif)
print('heuristic_alternatif =', h_alternatif)
print('path =', path_alt)
print('cost =', cost_alt)
print('expanded =', expanded_alt)

heuristic_alternatif = {'A': 5, 'B': 4, 'C': 10, 'D': 2, 'E': 0}
path = ['A', 'B', 'D', 'E']
cost = 8
expanded = ['A', 'B', 'D', 'E']


heuristic_alternatif = {'A': 5, 'B': 4, 'C': 10, 'D': 2, 'E': 0}
path = ['A', 'B', 'D', 'E']
cost = 8
expanded = ['A', 'B', 'D', 'E']

**Interpretasi:** ketika h(C) dinaikkan menjadi 10, A* memilih A-B-D-E dengan biaya 8, bukan rute termurah A-B-C-D-E dengan biaya 6. Ini menunjukkan heuristic yang terlalu tinggi dapat mengarahkan pencarian ke solusi yang tidak optimal. Eksperimen ini sengaja menguji heuristic buruk; nilai h(C)=10 melebihi biaya sebenarnya dari C ke E (3), sehingga tidak admissible.

## Evidence Log — ringkasan tiga percobaan

| Eksperimen | Perubahan | Metric / Output | Interpretasi |
|---|---|---|---|
| EXP-01 | Heuristic awal (A=5, B=4, C=3, D=2, E=0). | Path A-B-C-D-E; cost 6; expanded A, B, C, D, E (5 node). | Menjadi hasil dasar/pembanding. |
| EXP-02 | Semua heuristic diubah menjadi 0. | Path A-B-C-D-E; cost 6; expanded A, B, C, D, E (5 node). | Pada graph kecil ini, hasilnya sama dengan heuristic awal. h=0 membuat prioritas ditentukan oleh g(n), seperti Uniform Cost Search. |
| EXP-03 | h(C) diubah dari 3 menjadi 10; nilai lain tetap. | Path A-B-D-E; cost 8; expanded A, B, D, E (4 node). | Heuristic yang terlalu tinggi membuat A* melewatkan jalur yang lebih murah; node lebih sedikit tidak berarti solusi lebih baik. |

## Refleksi akhir

1. Apa hasil utama yang Anda amati?
heuristic awal dan h=0 menghasilkan path serta cost yang sama, sedangkan heuristic alternatif yang terlalu tinggi mengubah rute dan menghasilkan cost lebih besar.

2. Mengapa hasil tersebut terjadi menurut konsep di Main Deck?
A* memilih prioritas berdasarkan f(n)=g(n)+h(n). Perubahan h mengubah urutan prioritas node.

3. Apa limitation eksperimen ini?
graph hanya memiliki lima node. Hasil ini tidak otomatis berlaku untuk graph besar atau graph lain.

4. Eksperimen apa yang layak dilakukan berikutnya?
ubah satu bobot edge pada graph (misalnya biaya B ke D), lalu bandingkan path, cost, dan expanded dengan kondisi awal.